# 🎯 Regresión Logística - Clasificación desde CERO

## Objetivos
- Entender la función sigmoide
- Implementar regresión logística desde cero
- Binary Cross-Entropy Loss
- Gradient Descent para clasificación
- Evaluar con Accuracy, Precision, Recall

In [ ]:
# ==========================================
# CONFIGURACIÓN DEL ENTORNO
# ==========================================
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path

# Agregar el directorio raíz al path de manera robusta
project_root = Path.cwd().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Verificar que las utilidades se pueden importar
try:
    from utils.test_utils import check_answer
    from utils.plot_utils import plot_decision_boundary
    print("✅ Entorno configurado correctamente")
    print(f"📁 Raíz del proyecto: {project_root}")
except ImportError as e:
    print("❌ Error al importar utilidades")
    print("\n💡 Soluciones:")
    print("   1. Ejecuta 'pip install -e .' desde la raíz del proyecto")
    print("   2. O inicia Jupyter desde la raíz: cd ML-FROM-ZERO-PYTHON && jupyter notebook")
    print(f"\n🔍 Error detallado: {e}")
    raise

## 1. Teoría

### Función Sigmoide
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Convierte cualquier valor en un rango [0, 1], interpretable como probabilidad.

### El Modelo
$$z = w^T X + b$$
$$\hat{y} = \sigma(z)$$

### Binary Cross-Entropy Loss
$$J = -\frac{1}{m} \sum_{i=1}^{m} [y^{(i)} \log(\hat{y}^{(i)}) + (1-y^{(i)}) \log(1-\hat{y}^{(i)})]$$

### Gradientes
$$\frac{\partial J}{\partial w} = \frac{1}{m} X^T (\hat{y} - y)$$
$$\frac{\partial J}{\partial b} = \frac{1}{m} \sum (\hat{y} - y)$$

In [ ]:
# Visualizar función sigmoide
z = np.linspace(-10, 10, 100)
sigmoid = 1 / (1 + np.exp(-z))

plt.figure(figsize=(10, 6))
plt.plot(z, sigmoid, linewidth=2)
plt.axhline(y=0.5, color='r', linestyle='--', alpha=0.5)
plt.axvline(x=0, color='r', linestyle='--', alpha=0.5)
plt.xlabel('z')
plt.ylabel('σ(z)')
plt.title('Función Sigmoide')
plt.grid(True, alpha=0.3)
plt.show()

## 2. Implementación desde CERO

In [ ]:
class RegresionLogistica:
    """
    Regresión Logística implementada desde cero.
    """
    
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None
        self.loss_history = []
    
    def _sigmoid(self, z):
        """Función sigmoide"""
        return 1 / (1 + np.exp(-z))
    
    def fit(self, X, y):
        """Entrena el modelo"""
        X = np.array(X)
        y = np.array(y)
        
        if len(X.shape) == 1:
            X = X.reshape(-1, 1)
        
        n_samples, n_features = X.shape
        
        # Inicializar parámetros
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        # Gradient Descent
        for i in range(self.n_iterations):
            # Forward pass
            z = np.dot(X, self.weights) + self.bias
            y_pred = self._sigmoid(z)
            
            # Calcular gradientes
            dw = (1/n_samples) * np.dot(X.T, (y_pred - y))
            db = (1/n_samples) * np.sum(y_pred - y)
            
            # Actualizar parámetros
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db
            
            # Calcular loss (Binary Cross-Entropy)
            loss = self._binary_cross_entropy(y, y_pred)
            self.loss_history.append(loss)
            
            if (i + 1) % 100 == 0:
                print(f"Iteración {i+1}/{self.n_iterations}, Loss: {loss:.4f}")
        
        return self
    
    def _binary_cross_entropy(self, y_true, y_pred):
        """Binary Cross-Entropy Loss"""
        # Evitar log(0)
        epsilon = 1e-15
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
    
    def predict_proba(self, X):
        """Predice probabilidades"""
        X = np.array(X)
        if len(X.shape) == 1:
            X = X.reshape(-1, 1)
        
        z = np.dot(X, self.weights) + self.bias
        return self._sigmoid(z)
    
    def predict(self, X, threshold=0.5):
        """Predice clases (0 o 1)"""
        return (self.predict_proba(X) >= threshold).astype(int)
    
    def score(self, X, y):
        """Calcula accuracy"""
        y_pred = self.predict(X)
        return np.mean(y_pred == y)

## 3. Ejemplo: Clasificación Binaria

In [ ]:
# Generar datos sintéticos
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=200, n_features=2, n_redundant=0, 
                          n_informative=2, n_clusters_per_class=1, 
                          random_state=42)

# Visualizar
plt.figure(figsize=(10, 6))
plt.scatter(X[y==0, 0], X[y==0, 1], c='red', label='Clase 0', edgecolors='k')
plt.scatter(X[y==1, 0], X[y==1, 1], c='blue', label='Clase 1', edgecolors='k')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Datos de Clasificación Binaria')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Entrenar modelo
modelo = RegresionLogistica(learning_rate=0.1, n_iterations=1000)
modelo.fit(X, y)

# Evaluar
accuracy = modelo.score(X, y)
print(f"\nAccuracy: {accuracy:.4f}")

In [ ]:
# Visualizar frontera de decisión
plot_decision_boundary(X, y, modelo, title="Frontera de Decisión - Regresión Logística")
plt.show()

In [ ]:
# Curva de aprendizaje
plt.figure(figsize=(10, 6))
plt.plot(modelo.loss_history, linewidth=2)
plt.xlabel('Iteración')
plt.ylabel('Binary Cross-Entropy Loss')
plt.title('Curva de Aprendizaje')
plt.grid(True, alpha=0.3)
plt.show()

## 4. Métricas de Evaluación

In [ ]:
def calcular_metricas(y_true, y_pred):
    """Calcula Precision, Recall, F1"""
    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    
    accuracy = (TP + TN) / (TP + TN + FP + FN)
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'confusion_matrix': np.array([[TN, FP], [FN, TP]])
    }

y_pred = modelo.predict(X)
metricas = calcular_metricas(y, y_pred)

print("MÉTRICAS DE EVALUACIÓN:")
print(f"Accuracy:  {metricas['accuracy']:.4f}")
print(f"Precision: {metricas['precision']:.4f}")
print(f"Recall:    {metricas['recall']:.4f}")
print(f"F1-Score:  {metricas['f1']:.4f}")
print(f"\nMatriz de Confusión:")
print(metricas['confusion_matrix'])

## 🎯 Ejercicio: Implementa tu Regresión Logística

Crea una versión simplificada de la clase.

In [ ]:
# TU CÓDIGO AQUÍ
class MiRegresionLogistica:
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        pass
    
    def fit(self, X, y):
        pass
    
    def predict(self, X):
        pass

## 🎓 Resumen

- ✅ Función sigmoide
- ✅ Regresión logística desde cero
- ✅ Binary Cross-Entropy Loss
- ✅ Gradient Descent
- ✅ Métricas de clasificación

### Próximo: K-Nearest Neighbors (KNN)